# HEADS ML Pipeline (End-to-End)
This notebook runs the full pipeline:
1) Load + validate dataset
2) Feature engineering
3) Train Transformer Autoencoder → temporal anomaly score
4) Train GraphSAGE → relational anomaly score
5) Train XGBoost with SMOTE → final classifier + metrics

**Dataset:** `data/raw/cybersecurity.csv`


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from src.heads.data import load_csv
from src.heads.features import add_derived_features, TabularFeaturizer
from src.heads.temporal_ae import build_sequences, train_autoencoder, score_autoencoder
from src.heads.graph import build_graph
from src.heads.gnn import train_graphsage_linkpred, edge_anomaly_scores
from src.heads.smote import apply_smote
from src.heads.xgb import train_xgboost
from src.heads.config import HEADSConfig

import torch
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score


In [ ]:
DATA_PATH = Path('data/raw/cybersecurity.csv')
assert DATA_PATH.exists(), f"Missing {DATA_PATH}. Put your CSV there." 
df = load_csv(str(DATA_PATH))
df = add_derived_features(df)
df.head()


## 1) Quick validation\n

In [ ]:
required = ['timestamp','src_ip','dst_ip','src_port','dst_port','protocol','bytes_sent','bytes_received','user_agent','url','is_internal_traffic']
missing = [c for c in required if c not in df.columns]
missing


In [ ]:
print('Rows:', len(df))
print('Missing cells:', int(df.isna().sum().sum()))
if 'label' in df.columns:
    print('Attack rate:', float((df['label']==1).mean()))


## 2) Transformer Autoencoder (Temporal anomaly score)\nSequences are built per actor (`src_ip`). High reconstruction error ⇒ temporal anomaly.\n

In [ ]:
cfg = HEADSConfig(raw_csv=DATA_PATH)
cfg.model_dir.mkdir(parents=True, exist_ok=True)

ae_cols = ['src_port','dst_port','bytes_sent','bytes_received','bytes_total','hour','dow','is_web','is_internal_traffic']
df['is_internal_traffic'] = df['is_internal_traffic'].astype(int)

X_seq, y_last, idx_last = build_sequences(df, ae_cols, seq_len=cfg.seq_len, group_col='src_ip')
X_seq.shape


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ae = train_autoencoder(X_seq, cfg, device=device)
torch.save(ae.state_dict(), cfg.model_dir/'transformer_ae.pt')

temporal_scores = score_autoencoder(ae, X_seq, device=device)
df['temporal_score'] = 0.0
df.loc[idx_last, 'temporal_score'] = temporal_scores
df[['temporal_score']].describe()


## 3) GraphSAGE (Relational anomaly score)\nGraph: actor→resource/host/protocol edges. Link-pred training; low likelihood ⇒ anomaly.\n

In [ ]:
g_art = build_graph(df)
gnn = train_graphsage_linkpred(g_art.data, cfg, device=device)
torch.save(gnn.state_dict(), cfg.model_dir/'graphsage.pt')

import torch as _torch
edges = []
for _, r in df.iterrows():
    a = g_art.node_id_map[('actor', str(r['src_ip']))]
    b = g_art.node_id_map[('resource', str(r['dst_ip']))]
    edges.append((a,b))
edge_index = _torch.tensor(np.array(edges, dtype=np.int64).T, dtype=_torch.long)
df['relational_score'] = edge_anomaly_scores(gnn, g_art.data, edge_index, device=device)
df[['relational_score']].describe()


## 4) XGBoost fusion + SMOTE\nCombines tabular+text features with the two anomaly scores.\n

In [ ]:
if 'label' not in df.columns:
    print('No labels found → skipping supervised XGBoost. Scores are still available.')
else:
    feat = TabularFeaturizer(use_text=True, text_dim=64).fit(df)
    X_tab = feat.transform(df)
    X = np.hstack([X_tab, df[['temporal_score','relational_score']].to_numpy(float)])
    y = df['label'].astype(int).to_numpy()
    X_res, y_res = apply_smote(X, y, random_state=42)
    xgb = train_xgboost(X_res, y_res, cfg)
    df['xgb_proba'] = xgb.predict_proba(X)[:,1]
    df['prediction'] = (df['xgb_proba'] >= 0.5).astype(int)
    print('ROC-AUC:', roc_auc_score(y, df['xgb_proba']))
    print('PR-AUC:', average_precision_score(y, df['xgb_proba']))
    print(classification_report(y, df['prediction'], zero_division=0))


In [ ]:
out_path = Path('data/processed/scored_events.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)
out_path


## 5) Streamlit dashboard\nRun from the project root:\n```bash\nstreamlit run streamlit_app/app.py\n```\n